# Challenge five: Aero Alerts
## Automated weather alerts for airports
Goal: Demonstrate your ability to build an automated end-to-end service integrating BigQuery, Gemini, and Looker Studio

**Steps**
1. Define the functions
2. Create schema
3. Load raw data
4. Get forecast with external API
5. user gemini generate alert

**Create Scheduler**: hourly



In [75]:
import requests
from typing import Optional, List, Dict

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service (NWS) API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries for each time period,
        each containing:
            - 'name': Name of the forecast period (e.g., "Today", "Tonight")
            - 'startTime': ISO timestamp for the start of the forecast period
            - 'temperature': Temperature value
            - 'temperatureUnit': Temperature unit (e.g., "F" or "C")
            - 'windSpeed': Wind speed description
            - 'windDirection': Wind direction (e.g., "NW")
            - 'shortForecast': Short summary (e.g., "Partly Sunny")
            - 'detailedForecast': Full text forecast

        Returns None if data is unavailable or an error occurs.
    """
    headers = {
        'User-Agent': 'MyWeatherApp (doug@roitraining.com)',  # Replace with your actual email
        'Accept': 'application/geo+json'
    }

    # Step 1: Get metadata to find forecast URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    response = requests.get(points_url, headers=headers)

    if response.status_code != 200:
        print(f"Error fetching data from points endpoint: {response.status_code}")
        return None

    points_data = response.json()
    forecast_url = points_data['properties'].get('forecast')
    if not forecast_url:
        print("Forecast URL not found in response.")
        return None

    # Step 2: Fetch the forecast data
    forecast_response = requests.get(forecast_url, headers=headers)
    if forecast_response.status_code != 200:
        print(f"Error fetching forecast: {forecast_response.status_code}")
        return None

    forecast_data = forecast_response.json()
    periods = forecast_data['properties'].get('periods', [])

    if not periods:
        print("No forecast periods found in response.")
        return None

    # Return the full extended forecast
    extended_forecast = []
    for period in periods:
        extended_forecast.append({
            'name': period['name'],
            'startTime': period['startTime'],
            'temperature': str(period['temperature']),
            'temperatureUnit': period['temperatureUnit'],
            'windSpeed': period['windSpeed'],
            'windDirection': period['windDirection'],
            'shortForecast': period['shortForecast'],
            'detailedForecast': period['detailedForecast']
        })

    return extended_forecast


Step1: Create Schema bootcamp_hw5

In [72]:
%%bigquery
create schema if not exists bootcamp_hw5;


Query is running:   0%|          |

""


Step3: Check if raw table exist

In [73]:
%%bigquery
BEGIN
  -- Check if table exists
  IF EXISTS (
    SELECT 1
    FROM `bootcamp_hw5.INFORMATION_SCHEMA.TABLES`
    WHERE table_name = 'airports_raw'
  ) THEN
    -- Truncate table
    EXECUTE IMMEDIATE "TRUNCATE TABLE `bootcamp_hw5.airports_raw`";
  END IF;
END;

Query is running:   0%|          |

""


Step2: Create raw table from csv

In [74]:
%%bigquery

load data into bootcamp_hw5.airports_raw
from files (
format = 'csv',
field_delimiter = ',',
max_bad_records = 0,
uris = ['gs://labs.roitraining.com/data-to-ai-workshop/airports.csv']
)

Query is running:   0%|          |

""


Step3: Create dataframe from table

In [76]:
%%bigquery df
SELECT *
FROM `bootcamp_hw5.airports_raw`
where iso_country='US' and type='large_airport'

Query is running:   0%|          |

Downloading:   0%|          |

Step4: Generate forecast from table

In [80]:
import json
df['forecast'] = df.apply(
    lambda row: get_extended_weather_forecast(row['latitude_deg'], row['longitude_deg']),
    axis=1
)
df['forecast'] = df['forecast'].apply(lambda x: json.dumps(x))

Step5: Save forecast to new table

In [82]:
from google.cloud import bigquery

client = bigquery.Client()
table_id = "bootcamp_hw5.airports_forecast"
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)
job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()

LoadJob<project=qwiklabs-gcp-03-d68060dab4e3, location=US, id=8e41b5f6-8d8e-42ad-855d-1e7d54ce749c>

Step6: create model link

In [83]:
%%bigquery
CREATE OR REPLACE MODEL `bootcamp_hw5.gemini_model_20`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'gemini-2.0-flash');

Query is running:   0%|          |

""


Step7: generate alert with gemini model

In [90]:
%%bigquery
create or replace table bootcamp_hw5.airports_alert as
SELECT
  ml_generate_text_llm_result AS generated_text,
  *
FROM ML.GENERATE_TEXT(
  MODEL `bootcamp_hw5.gemini_model_20`,
  (
    SELECT *, CONCAT(
          """You are a weather alert generator. You receive a list of weather forecasts in JSON format.

            Task:
            - Only generate an alert for very bad weather (rain, snow, thunderstorms, high wind, extreme temperature, etc.)
            - Ignore normal or pleasant weather
            - The total alert text less than 200 characters
            - first line is city name and airport name, follow the format: city, airport name, for example: Phoenix, Phoenix Sky Harbor International Airport
            - the first line and other lines should be seprated with an empty line
            - for alert Include the day/time, and issue (e.g., "Today: Light Rain, high chance 80%")
            - Do not include JSON or extra formatting

             """,
      " airport name: ", name,
      " country: ", iso_country,
      " city: ",  municipality,
      "forecase: ", forecast
    ) AS prompt
    FROM `bootcamp_hw5.airports_forecast`
  ),
  STRUCT(
    TRUE  AS flatten_json_output,
    1024 as max_output_tokens
    )
    ) ;

Query is running:   0%|          |

""
